# Jaguar Data Ingestion Pipeline (0226)

This notebook runs ingestion for the 25_02_2026 camera-trap dataset with a review-first workflow:
1. Load CSV/PPTX into a grouped FiftyOne dataset
2. Sample video frames
3. Run SAM3 segmentation and **tag** low-quality/filtered samples (no deletion)
4. Compute embeddings
5. Add train/val/test split fields + tags
6. Run deduplication (flags + tags, no deletion)
7. Publish full FiftyOne dataset to private Hugging Face: `jaguars_camera_trap_0226_fiftyone`
8. After manual review, publish 4 derived private datasets to Hugging Face

Manual review happens on another machine with FiftyOne UI, then this notebook exports curated variants.

In [1]:
# Should install the package in editable mode to reflect recent changes
%pip install -e ../

Obtaining file:///sc/home/philipp.kolbe/JID/camera-trap-footage
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jaguars (pyproject.toml) ... done
  Created wheel for jaguars: filename=jaguars-0.1.0-0.editable-py3-none-any.whl size=3246 sha256=ef79a3f79b560e33a4066d4119a27eae0450fe82e2870a547e17c6dc9e7ae3bb
  Stored in directory: /tmp/pip-ephem-wheel-cache-45sb90kw/wheels/fb/73/df/dfc2bfca6660ef97fbfd8e766c6c3a255e3c7b2656a8649611
Successfully built jaguars
  Attempting uninstall: jaguars
    Found existing installation: jaguars 0.1.0
    Uninstalling jaguars-0.1.0:
      Successfully uninstalled jaguars-0.1.0
Note: you may need to restart the kernel to use updated packages.


## Setup and Imports

In [1]:
import os
import sys
from pathlib import Path
import logging
import numpy as np
from PIL import Image

# Add src to path if needed
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import fiftyone as fo
from fiftyone import ViewField as F
from fiftyone.utils.huggingface import push_to_hub
from huggingface_hub import login

from jaguars.common.logging_utils import setup_logger
from jaguars.common.config import JID_MASTER_DATASET
from jaguars.ingestion.check_dataset_state import check_dataset_state
from jaguars.ingestion.loaders.csv_loader import ingest_csv_labels
from jaguars.ingestion.loaders.pptx_loader import ingest_pptx_slides
from jaguars.ingestion.processing.sample import run_processing as run_sample
from jaguars.ingestion.processing.add_embeddings import run_processing as run_add_embeddings
from jaguars.ingestion.processing.add_multi_backbone_embeddings import run_processing as run_multi_embeddings
from jaguars.ingestion.processing.split import run_processing as run_split
from jaguars.ingestion.processing.deduplicate import run_processing as run_deduplicate
from jaguars.ingestion.export.export import run_processing as run_export
from jaguars.segmentation.SAM3 import run_processing as run_sam3
from jaguars.ingestion.processing.segmentation import filter_by_count, filter_unexpected_segmentations

# Setup logging
logger = setup_logger("ingestion_notebook", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Configuration

Set paths and parameters for the ingestion pipeline.

In [ ]:
# Data paths
INPUT_DIR = project_root / "data" / "raw" / "25_02_2026"
LABELS_CSV = INPUT_DIR / "labels.csv"  # new CSV in same name
PPTX_PATHS = [
    INPUT_DIR / "Copy of CAMERA TRAP ID GUIDE 2_1_26.pptx",
    INPUT_DIR / "Copy of CAMERA TRAP ID GUIDE 26_1_25.pptx",
]

# FiftyOne dataset configuration
DATASET_NAME = JID_MASTER_DATASET  # grouped dataset
PPTX_MEDIA_DIR = None
PPTX_DETECTIONS_FIELD = "pptx_detections"

# CSV ingestion parameters
AUTO_MATCH_MISSING = True
MATCH_THRESHOLD = 0.95
SUGGEST_THRESHOLD = 0.80

# Processing parameters
SEGMENTATION_FIELD = "sam3_segmentations"
SEGMENTATION_PROMPT = "jaguar"
SEGMENTATION_TAG_COUNT = "filter_count"
SEGMENTATION_TAG_QUALITY = "filter_quality"
EMBEDDING_BATCH_SIZE = 32  # safer default for A40/large datasets
MULTI_BACKBONE_BATCH_SIZE = 32  # multi-backbone is heavier on GPU memory

# Split parameters
RUN_SPLIT = True
TAG_SPLITS = True  # adds tags train/val/test
SPLIT_FIELD = "closed_set_split"

# Deduplication parameters
RUN_DEDUP = True
DEDUP_SIMILARITY_THRESHOLD = 0.98
DEDUP_FIELD = "is_duplicate"
DEDUP_TAG = "duplicate"

# Export configuration
EXPORT_BASE_DIR = project_root / "data" / "intermediate" / "v1" / "fo_jaguars" / "exports"
EXPORT_TARGETS = ["disk"]

# Hugging Face private repos (set HF_TOKEN env var before pushing)
HF_REPO_FULL_FIFTYONE = "jaguars_camera_trap_0226_fiftyone"
HF_REPO_SEGMENTED_DEDUP = "jaguars_camera_trap_0226"
HF_REPO_SEGMENTED_WITH_DUP = "jaguars_camera_trap_0226_duplicates"
HF_REPO_RAW_DEDUP = "jaguars_camera_trap_0226_raw"
HF_REPO_MASK_BINARY_DEDUP = "jaguars_camera_trap_0226_masked"

# Review gate
PUBLISH_DERIVED_AFTER_REVIEW = False  # keep False until manual review on another machine is complete

# Runtime
OVERWRITE_DATASET = True
VERBOSE = True

print("Configuration:")
print(f"  Input directory: {INPUT_DIR}")
print(f"  Labels CSV: {LABELS_CSV}")
print("  PPTX candidates:")
for p in PPTX_PATHS:
    print(f"    - {p}")
print(f"  Dataset name: {DATASET_NAME}")
print(f"  Embedding batch size: {EMBEDDING_BATCH_SIZE}")
print(f"  Multi-backbone batch size: {MULTI_BACKBONE_BATCH_SIZE}")
print(f"  Export base dir: {EXPORT_BASE_DIR}")
print(f"  Full FiftyOne HF repo: {HF_REPO_FULL_FIFTYONE}")
print(f"  Split field: {SPLIT_FIELD}")
print(f"  Publish derived after review: {PUBLISH_DERIVED_AFTER_REVIEW}")
print(f"  Overwrite dataset: {OVERWRITE_DATASET}")

Configuration:
  Input directory: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026
  Labels CSV: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/labels.csv
  PPTX candidates:
    - /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/Copy of CAMERA TRAP ID GUIDE 2_1_26.pptx
    - /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/Copy of CAMERA TRAP ID GUIDE 26_1_25.pptx
  Dataset name: JID_Master_Dataset
  Embedding batch size: 4
  Multi-backbone batch size: 4
  Export base dir: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/intermediate/v1/fo_jaguars/exports
  Full FiftyOne HF repo: jaguars_camera_trap_0226_fiftyone
  Split field: closed_set_split
  Publish derived after review: False
  Overwrite dataset: True


In [4]:
check_dataset_state(DATASET_NAME)

✓ Dataset 'JID_Master_Dataset' exists

DATASET OVERVIEW
Total samples: 30
Group field: group
Image samples: 30
Video samples: 509

FIELDS
  id: fiftyone.core.fields.ObjectIdField
  filepath: fiftyone.core.fields.StringField
  tags: fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
  metadata: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
  created_at: fiftyone.core.fields.DateTimeField
  last_modified_at: fiftyone.core.fields.DateTimeField
  group: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.groups.Group)
  source_type: fiftyone.core.fields.StringField
  source: fiftyone.core.fields.StringField
  csv_source: fiftyone.core.fields.StringField
  jaguar_id: fiftyone.core.fields.StringField
  ground_truth: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
  site: fiftyone.core.fields.StringField
  cam: fiftyone.core.fields.StringField
  sighting_id: fiftyone.core.fields.StringField
  date: fiftyone.core.

{'exists': True,
 'images_count': 30,
 'videos_count': 509,
 'csv_loaded': True,
 'pptx_loaded': False,
 'frames_sampled': 0}

In [ ]:
# Remove existing dataset if needed
if OVERWRITE_DATASET and fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
    print(f"✓ Deleted existing dataset '{DATASET_NAME}'")

✓ Deleted existing dataset 'JID_Master_Dataset'


In [3]:
if fo.dataset_exists(DATASET_NAME):
    dataset = fo.load_dataset(DATASET_NAME)
    print(f"✓ Loaded existing dataset '{DATASET_NAME}' with {len(dataset)} samples")

✓ Loaded existing dataset 'JID_Master_Dataset' with 5397 samples


## Step 1: Data Loading

Load data from CSV labels and PPTX files into FiftyOne.

### Step 1a: CSV Labels Ingestion

Ingest video metadata and labels from CSV file.

In [7]:
print("=" * 70)
print("STEP 1a: CSV Labels Ingestion")
print("=" * 70)

# Ingest CSV labels
csv_kwargs = {
    "input_dir": INPUT_DIR,
    "dataset_name": DATASET_NAME,
    "auto_match_missing": AUTO_MATCH_MISSING,
    "match_threshold": MATCH_THRESHOLD,
    "suggest_threshold": SUGGEST_THRESHOLD,
    "generate_report": True,
}

if LABELS_CSV and LABELS_CSV.exists():
    csv_kwargs["input_csv"] = LABELS_CSV

print(f"Loading from: {INPUT_DIR}")
dataset = ingest_csv_labels(**csv_kwargs)

print("\n✓ CSV ingestion completed!")
print(f"  Dataset: {dataset.name}")
print(f"  Total samples: {len(dataset)}")
print(f"  Group field: {dataset.group_field}")

# Access image and video slices
images_view = dataset.select_group_slices("image")
videos_view = dataset.select_group_slices("video")

print(f"  Image samples: {len(images_view)}")
print(f"  Video samples: {len(videos_view)}")

STEP 1a: CSV Labels Ingestion
Loading from: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026
16:47:34 - jid_logger.ingestion.loaders.csv_loader - INFO - Cleaning labels using CSV cleaning pipeline: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/labels.csv


16:47:34 - jid_logger.ingestion.loaders.csv_loader - INFO - Loaded 637 rows with columns: CAMERA TRAP SITE , DATE, LATITUDE, LONGITUDE, CAMERA ID, CAM , JAGUAR ID, SEX, LOCATION, CAMERA MODEL, TIME, TEMP C, Files Name, NOTES/ ERRORS, Gaia lat, Gaia long
16:47:34 - jid_logger.ingestion.loaders.csv_loader - INFO - All optional columns present
16:47:34 - jid_logger.ingestion.loaders.csv_loader - INFO - Filling 1 missing JAGUAR IDs
16:47:34 - jid_logger.ingestion.loaders.csv_loader - INFO - Assigned 1 new JAGUAR IDs
16:47:35 - jid_logger.ingestion.loaders.csv_loader - INFO - Parsed 635/637 DATE entries
16:47:35 - jid_logger.ingestion.loaders.csv_loader - INFO - Split 114 rows with multifile sightings into 839 total rows
16:47:35 - jid_logger.ingestion.loaders.csv_loader - INFO - Found 2 duplicate entries based on (JAGUAR ID, CAMERA TRAP SITE, CAM, ORIGINAL FILE NAME)
16:47:35 - jid_logger.ingestion.loaders.csv_loader - INFO - Duplicate entries:
    CAMERA TRAP SITE        DATE     LATITUDE

Converting AVI to MP4: 100%|██████████| 49/49 [00:00<00:00, 32142.77it/s]


16:47:42 - jid_logger.ingestion.loaders.csv_loader - INFO - Filtering out 0 rows with empty FILE PATHs


INFO:jid_logger.ingestion.loaders.csv_loader:Filtering out 0 rows with empty FILE PATHs


...

16:47:48 - jid_logger.ingestion.loaders.csv_loader - INFO - Added 539 samples to grouped dataset (30 images, 509 videos)


INFO:jid_logger.ingestion.loaders.csv_loader:Added 539 samples to grouped dataset (30 images, 509 videos)


16:47:48 - jid_logger.ingestion.loaders.csv_loader - INFO - CSV Ingestion Summary: {
  "dataset_name": "JID_Master_Dataset",
  "is_grouped": true,
  "slices": [
    "image",
    "video"
  ],
  "csv_source": "/sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/labels.csv",
  "new_image_samples_added": 30,
  "new_video_samples_added": 509,
  "total_image_samples_in_dataset": 30,
  "total_video_samples_in_dataset": 509,
  "total_samples_in_dataset": 539,
  "cleaned_rows_total": 836
}


INFO:jid_logger.ingestion.loaders.csv_loader:CSV Ingestion Summary: {
  "dataset_name": "JID_Master_Dataset",
  "is_grouped": true,
  "slices": [
    "image",
    "video"
  ],
  "csv_source": "/sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/labels.csv",
  "new_image_samples_added": 30,
  "new_video_samples_added": 509,
  "total_image_samples_in_dataset": 30,
  "total_video_samples_in_dataset": 509,
  "total_samples_in_dataset": 539,
  "cleaned_rows_total": 836
}



✓ CSV ingestion completed!
  Dataset: JID_Master_Dataset
  Total samples: 30
  Group field: group
  Image samples: 30
  Video samples: 509


### Step 1b: PPTX Ingestion

Extract and ingest jaguar reference images from PowerPoint presentation.

In [8]:
print("=" * 70)
print("STEP 1b: PPTX Ingestion")
print("=" * 70)

ingested_pptx = []
for pptx_path in PPTX_PATHS:
    if pptx_path.exists():
        print(f"Loading PPTX: {pptx_path}")
        dataset = ingest_pptx_slides(
            pptx_path=pptx_path,
            dataset_name=DATASET_NAME,
            media_dir=PPTX_MEDIA_DIR,
            detections_field=PPTX_DETECTIONS_FIELD,
            generate_report=True,
        )
        ingested_pptx.append(pptx_path.name)
    else:
        print(f"⚠ PPTX file not found: {pptx_path}")

if ingested_pptx:
    print("\n✓ PPTX ingestion completed for:")
    for name in ingested_pptx:
        print(f"  - {name}")
else:
    print("⚠ No PPTX ingested")

# Refresh views
images_view = dataset.select_group_slices("image")
videos_view = dataset.select_group_slices("video")
print(f"  Total samples: {len(dataset)}")
print(f"  Image samples: {len(images_view)}")
print(f"  Video samples: {len(videos_view)}")
print(f"  PPTX-derived samples: {len(images_view.match_tags('pptx'))}")

STEP 1b: PPTX Ingestion
Loading PPTX: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/Copy of CAMERA TRAP ID GUIDE 2_1_26.pptx
16:47:48 - jid_logger.ingestion.loaders.pptx_loader - INFO - Loading PowerPoint from /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/Copy of CAMERA TRAP ID GUIDE 2_1_26.pptx


INFO:jid_logger.ingestion.loaders.pptx_loader:Loading PowerPoint from /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/Copy of CAMERA TRAP ID GUIDE 2_1_26.pptx


16:47:48 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset


INFO:jid_logger.common.fiftyone_utils:Loading existing dataset: JID_Master_Dataset


16:47:48 - jid_logger.ingestion.loaders.pptx_loader - INFO - Loading PowerPoint from /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/Copy of CAMERA TRAP ID GUIDE 2_1_26.pptx


...

16:47:53 - jid_logger.ingestion.loaders.pptx_loader - INFO - Added 11 samples from PPTX


INFO:jid_logger.ingestion.loaders.pptx_loader:Added 11 samples from PPTX


16:47:53 - jid_logger.ingestion.loaders.pptx_loader - INFO - PPTX Ingestion Summary: {
  "dataset_name": "JID_Master_Dataset",
  "pptx_source": "/sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/Copy of CAMERA TRAP ID GUIDE 26_1_25.pptx",
  "media_dir": "/sc/home/philipp.kolbe/fiftyone/JID_Master_Dataset/media/pptx",
  "slides_total": 123,
  "slides_with_images": 116,
  "images_extracted": 243,
  "new_samples_added": 11,
  "total_samples_in_dataset": 312,
  "unique_jaguar_ids": 115,
  "unique_sites": 7
}


INFO:jid_logger.ingestion.loaders.pptx_loader:PPTX Ingestion Summary: {
  "dataset_name": "JID_Master_Dataset",
  "pptx_source": "/sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/25_02_2026/Copy of CAMERA TRAP ID GUIDE 26_1_25.pptx",
  "media_dir": "/sc/home/philipp.kolbe/fiftyone/JID_Master_Dataset/media/pptx",
  "slides_total": 123,
  "slides_with_images": 116,
  "images_extracted": 243,
  "new_samples_added": 11,
  "total_samples_in_dataset": 312,
  "unique_jaguar_ids": 115,
  "unique_sites": 7
}



✓ PPTX ingestion completed for:
  - Copy of CAMERA TRAP ID GUIDE 2_1_26.pptx
  - Copy of CAMERA TRAP ID GUIDE 26_1_25.pptx
  Total samples: 312
  Image samples: 312
  Video samples: 509
  PPTX-derived samples: 0


### Visualize Loaded Dataset

Launch FiftyOne App to inspect the ingested data.

In [ ]:
# start fiftyone
session = fo.launch_app(dataset)
print("✓ FiftyOne UI launched")

print("Manual review should be done after pushing to HF on your review machine.")
print(f"Dataset ready for push: {DATASET_NAME}")

## Step 2: Video Frame Sampling

Extract frames from videos at regular intervals.

In [9]:
print("=" * 70)
print("STEP 2: Video Frame Sampling")
print("=" * 70)

sampling_results = run_sample(
    dataset_name=DATASET_NAME,
    verbose=VERBOSE
)

STEP 2: Video Frame Sampling
16:48:44 - jid_logger.ingestion.processing.sample - INFO - Starting video frame sampling for dataset: JID_Master_Dataset


INFO:jid_logger.ingestion.processing.sample:Starting video frame sampling for dataset: JID_Master_Dataset


16:48:44 - jid_logger.ingestion.processing.sample - INFO - Sampling strategy: 1.0 fps for first 5.0 seconds, then 0.16666666666666666 fps for remainder


INFO:jid_logger.ingestion.processing.sample:Sampling strategy: 1.0 fps for first 5.0 seconds, then 0.16666666666666666 fps for remainder


16:48:45 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset


...

INFO:jid_logger.ingestion.processing.sample:Added 5085 sampled frames from 509 videos


17:47:04 - jid_logger.ingestion.processing.sample - INFO - Sampling Summary: {
  "videos_processed": 509,
  "frames_added": 5085,
  "total_image_samples": 5397,
  "total_video_samples": 509,
  "total_samples": 5397
}


INFO:jid_logger.ingestion.processing.sample:Sampling Summary: {
  "videos_processed": 509,
  "frames_added": 5085,
  "total_image_samples": 5397,
  "total_video_samples": 509,
  "total_samples": 5397
}


17:47:04 - jid_logger.ingestion.processing.sample - INFO - Video frame sampling completed successfully.


INFO:jid_logger.ingestion.processing.sample:Video frame sampling completed successfully.


In [8]:
# Refresh dataset
dataset.reload()
images_view = dataset.select_group_slices("image")
print(f"  Total image samples (after sampling): {len(images_view)}")

  Total image samples (after sampling): 2454


## Step 3: Segmentation with SAM3

Detect and segment jaguars in images using SAM3.

In [5]:
print("=" * 70)
print("STEP 3: Segmentation + Tag-Only Filtering")
print("=" * 70)

# 1) Run SAM3 segmentation and store detections; do not delete anything
dataset = run_sam3(
    dataset_name=DATASET_NAME,
    prompt=SEGMENTATION_PROMPT,
    threshold=0.5,
    mask_threshold=0.5,
    output_field=SEGMENTATION_FIELD,
    verbose=VERBOSE,
)

# 2) Tag segmentation issues; do not delete anything
n_count = filter_by_count(
    dataset=dataset,
    segmentation_field=SEGMENTATION_FIELD,
    expected_count=1,
    tag=SEGMENTATION_TAG_COUNT,
)
n_quality = filter_unexpected_segmentations(
    dataset=dataset,
    segmentation_field=SEGMENTATION_FIELD,
    min_confidence=0.5,
    min_area_rel=0.01,
    max_area_rel=1.0,
    tag=SEGMENTATION_TAG_QUALITY,
)

dataset.reload()
print("\n✓ Segmentation completed (tag-only mode)")
print(f"  Tagged count issues: {n_count}")
print(f"  Tagged quality issues: {n_quality}")
print(f"  Samples with segmentation field: {len(dataset.exists(SEGMENTATION_FIELD))}")
print("  No samples were deleted in this step.")

STEP 3: Segmentation + Tag-Only Filtering
18:31:27 - jid_logger.segmentation.SAM3 - INFO - Starting SAM3 segmentation for dataset: JID_Master_Dataset
18:31:27 - jid_logger.segmentation.SAM3 - INFO - Parameters: prompt='jaguar', threshold=0.5, mask_threshold=0.5
18:31:27 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset
18:31:27 - jid_logger.segmentation.SAM3 - INFO - Dataset is grouped. Selecting 'image' slice.
18:31:27 - jid_logger.segmentation.SAM3 - INFO - Found 5397 samples to process
18:31:27 - jid_logger.segmentation.SAM3 - INFO - Registering SAM3 zoo model...
18:31:27 - jid_logger.segmentation.SAM3 - INFO - Loading SAM3 model on device: cuda


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

18:32:01 - jid_logger.segmentation.SAM3 - INFO - Applying model to 5397 samples...
 100% |███████████████| 5397/5397 [48.8m elapsed, 0s remaining, 1.8 samples/s]    
19:20:49 - jid_logger.segmentation.SAM3 - INFO - Saving view...
19:20:50 - jid_logger.segmentation.SAM3 - INFO - SAM3 segmentation output saved to field 'sam3_segmentations'
19:20:54 - jid_logger.ingestion.processing.segmentation - INFO - Tagged 2581 samples with 'filter_count' (detection count != 1)
19:20:54 - jid_logger.ingestion.processing.segmentation - INFO - Tagged 110 samples with 'filter_quality' (quality issues)

✓ Segmentation completed (tag-only mode)
  Tagged count issues: 2581
  Tagged quality issues: 110
  Samples with segmentation field: 5397
  No samples were deleted in this step.


## Step 4: Compute Embeddings

Generate embeddings for jaguar detections using a pre-trained model.

In [ ]:
print("=" * 70)
print("STEP 4: Embedding Computation")
print("=" * 70)

mask_embedding_results = run_add_embeddings(
    dataset_name=DATASET_NAME,
    patches_field=SEGMENTATION_FIELD,  # Use SAM3 segmentations
    mask_field="mask",  # SAM3 adds mask field
    batch_size=EMBEDDING_BATCH_SIZE,
    verbose=VERBOSE,
)

print("\n✓ Segmented mask embedding computation completed!")

STEP 4: Embedding Computation
22:26:08 - jid_logger.ingestion.processing.add_embeddings - INFO - Starting embedding computation for dataset: JID_Master_Dataset
22:26:08 - jid_logger.ingestion.processing.add_embeddings - INFO - Model: hf-hub:BVRA/MegaDescriptor-L-384
22:26:08 - jid_logger.ingestion.processing.add_embeddings - INFO - Target Field: embeddings_BVRA_MegaDescriptor_L_384
22:26:08 - jid_logger.ingestion.processing.add_embeddings - INFO - Processing patches from field: sam3_segmentations
22:26:08 - jid_logger.ingestion.processing.add_embeddings - INFO - Using masks from field: mask
22:26:08 - jid_logger.ingestion.processing.add_embeddings - INFO - Batch Size: 4
22:26:08 - jid_logger.ingestion.processing.add_embeddings - INFO - Selected 'image' slice for processing. 5397 samples found.
22:26:08 - jid_logger.ingestion.processing.add_embeddings - INFO - Gathering detection patches from field 'sam3_segmentations'...
22:26:31 - jid_logger.ingestion.processing.add_embeddings - INFO 

Computing embeddings:  29%|██▉       | 940/3240 [01:00<02:44, 13.95it/s]

In [ ]:
full_image_embedding_results = run_add_embeddings(
    dataset_name=DATASET_NAME,
    batch_size=EMBEDDING_BATCH_SIZE,
    verbose=VERBOSE,
)

print("\n✓ Full image embedding computation completed!")

14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Starting embedding computation for dataset: JID_Master_Dataset
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Model: hf-hub:BVRA/MegaDescriptor-L-384
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Target Field: embeddings_BVRA_MegaDescriptor_L_384
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Batch Size: 32
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Selected 'image' slice for processing. 1171 samples found.
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Gathering inputs...
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Using device: cuda
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Loading model hf-hub:BVRA/MegaDescriptor-L-384...
14:05:04 - jid_logger.ingestion.processing.add_embeddings - INFO - Model loaded in 3.81 seconds
14:05:04 - jid_logger.ingestion.processing.add_embe

14:06:16 - jid_logger.ingestion.processing.add_embeddings - INFO - Computed 1171 embeddings in 72.24 seconds
14:06:16 - jid_logger.ingestion.processing.add_embeddings - INFO - Saving embeddings to field 'embeddings_BVRA_MegaDescriptor_L_384'...


14:06:17 - jid_logger.ingestion.processing.add_embeddings - INFO - Embedding computation completed successfully.

✓ Full image embedding computation completed!


### Step 4b: Further Embeddings

In [8]:
# reload module run_multi_embeddings
import importlib
import jaguars.ingestion.processing.add_multi_backbone_embeddings as run_multi_embeddings_module
importlib.reload(run_multi_embeddings_module)
from jaguars.ingestion.processing.add_multi_backbone_embeddings import run_processing as run_multi_embeddings

In [ ]:
print("=" * 70)
print("STEP 4b: Multi-Backbone Embeddings (CACHED)")
print("=" * 70)

# Compute embeddings for all backbones (one-time)
multi_emb_results = run_multi_embeddings(
    dataset_name=DATASET_NAME,
    patches_field=SEGMENTATION_FIELD,
    batch_size=MULTI_BACKBONE_BATCH_SIZE,
    device="cuda",
    overwrite=False,  # Skip if already computed
    verbose=VERBOSE,
)

print("\n✓ Multi-backbone embeddings completed!")
dataset.reload()

STEP 4b: Multi-Backbone Embeddings (CACHED)
01:45:13 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - Computing embeddings for 8 backbones
01:45:13 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   - hf-hub:BVRA/MegaDescriptor-L-384 → embeddings_BVRA_MegaDescriptor_L_384
01:45:13 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   - hf-hub:BVRA/MegaDescriptor-B-224 → embeddings_BVRA_MegaDescriptor_B_224
01:45:13 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   - vit_large_patch14_dinov2.lvd142m → embeddings_DINOv2_Large
01:45:13 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   - vit_base_patch14_dinov2.lvd142m → embeddings_DINOv2_Base
01:45:13 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   - resnet50 → embeddings_ResNet50
01:45:13 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   - convnextv2_base.fcmae_ft_in22k

Computing vit_large_patch16_dinov3.lvd1689m embeddings: 100%|██████████| 37/37 [01:31<00:00,  2.48s/it]

01:47:05 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   Storing 1171 embeddings...


01:48:05 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   ✓ Computed embeddings for 1171 patches
01:48:05 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - 
01:48:05 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - Processing backbone: vit_base_patch16_dinov3.lvd1689m
01:48:05 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   Embedding field: embeddings_DINOv3_Base
01:48:05 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - ======================================================================
01:48:05 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   Loading model...


Loading vit_base_patch16_dinov3.lvd1689m model...


...

Computing vit_base_patch16_dinov3.lvd1689m embeddings: 100%|██████████| 37/37 [00:48<00:00,  1.32s/it]

01:49:12 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   Storing 1171 embeddings...


01:50:15 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   ✓ Computed embeddings for 1171 patches
01:50:15 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - 
01:50:15 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - Multi-backbone embedding computation completed!
01:50:15 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - ======================================================================



✓ Multi-backbone embeddings completed!


NameError: name 'dataset' is not defined

## Step 5: Split Dataset

Create train/val/test splits and optionally tag samples.

In [13]:
print("=" * 70)
print("STEP 5: Dataset Splitting")
print("=" * 70)

if RUN_SPLIT:
    split_results = run_split(
        dataset_name=DATASET_NAME,
        add_closed_set=True,
        add_open_set=False,
        tag_by="closed" if TAG_SPLITS else None,
        verbose=VERBOSE,
    )
    print("\n✓ Split completed!")
else:
    print("Skipping split step")
    split_results = None

STEP 5: Dataset Splitting
14:08:50 - jid_logger.ingestion.processing.split - INFO - Starting split processing for dataset: JID_Master_Dataset
14:08:50 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset
14:08:50 - jid_logger.ingestion.processing.split - INFO - Filtering dataset to 'image' group slice only
14:08:50 - jid_logger.ingestion.processing.split - INFO - Working with 1171 samples from 'image' slice
14:08:50 - jid_logger.ingestion.processing.split - INFO - Using ID field: jaguar_id
14:08:50 - jid_logger.ingestion.processing.split - INFO - Applying Closed-Set Split...


Closed-set split: 100%|██████████| 76/76 [00:00<00:00, 284.95it/s]


14:08:51 - jid_logger.ingestion.processing.split - INFO - Verifying Closed-Set integrity...
14:08:51 - jid_logger.ingestion.processing.split - INFO - Counts for closed_set_split: {'val': 120, 'test': 105, 'train': 946}
14:08:51 - jid_logger.ingestion.processing.split - INFO - Tagging samples by closed-set split...
14:08:52 - jid_logger.ingestion.processing.split - INFO - Tagged 946 samples with 'train'
14:08:52 - jid_logger.ingestion.processing.split - INFO - Tagged 120 samples with 'val'
14:08:52 - jid_logger.ingestion.processing.split - INFO - Tagged 105 samples with 'test'

✓ Split completed!


## Step 6: Deduplicate Images

Flag near-duplicate images (no segmentation, full image only).

In [20]:
# reload run_deduplicate package so we dont have to restart
import importlib
import jaguars.ingestion.processing.deduplicate as dedup_module
importlib.reload(dedup_module)

from jaguars.ingestion.processing.deduplicate import run_processing as run_deduplicate

In [ ]:
print("=" * 70)
print("STEP 6: Deduplication")
print("=" * 70)

if RUN_DEDUP:
    dedup_results = run_deduplicate(
        dataset_name=DATASET_NAME,
        similarity_threshold=DEDUP_SIMILARITY_THRESHOLD,
        is_duplicate_field=DEDUP_FIELD,
        duplicate_tag=DEDUP_TAG,
        verbose=VERBOSE,
    )
    dataset.reload()
    img_view = dataset.select_group_slices("image") if dataset.group_field else dataset
    print("\n✓ Deduplication completed!")
    print(f"  Duplicates flagged: {len(img_view.match(F(DEDUP_FIELD) == True))}")
else:
    print("Skipping deduplication step")

STEP 6: Deduplication
14:18:26 - jid_logger.ingestion.processing.deduplicate - INFO - Starting deduplication for dataset: JID_Master_Dataset
14:18:26 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset
14:18:26 - jid_logger.ingestion.processing.deduplicate - INFO - Clearing previous duplicate markings...
14:18:27 - jid_logger.ingestion.processing.deduplicate - INFO - Previous duplicate markings cleared
14:18:27 - jid_logger.ingestion.processing.deduplicate - INFO - Using embeddings field: embeddings_BVRA_MegaDescriptor_L_384
14:18:27 - jid_logger.ingestion.processing.deduplicate - INFO - Embeddings field 'embeddings_BVRA_MegaDescriptor_L_384' already exists, using existing embeddings
14:18:27 - jid_logger.ingestion.processing.deduplicate - INFO - Running FiftyOne brain near duplicate detection (threshold=0.980)
14:18:28 - jid_logger.ingestion.processing.deduplicate - DEBUG - Marked 69808241341041adb61e5fe3 as duplicate of 6980821c341041adb61e5d89 (s

Name:        JID_Master_Dataset
Media type:  group
Group slice: image
Num groups:  1171
Persistent:  True
Tags:        []
Sample fields:
    id:                                   fiftyone.core.fields.ObjectIdField
    filepath:                             fiftyone.core.fields.StringField
    tags:                                 fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                             fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
    created_at:                           fiftyone.core.fields.DateTimeField
    last_modified_at:                     fiftyone.core.fields.DateTimeField
    group:                                fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.groups.Group)
    source_type:                          fiftyone.core.fields.StringField
    source:                               fiftyone.core.fields.StringField
    csv_source:                           fiftyone.core.fields.String

## Step 7: Publish to Hugging Face (Review-First Workflow)

1. Publish full FiftyOne dataset (all samples + tags + detections + segmentations + dedup flags) to private repo:
   `jaguars_camera_trap_0226_fiftyone`
2. Manually review/correct on another machine in FiftyOne UI
3. Publish 4 derived private datasets:
   - `jaguars_camera_trap_0226` (segmented RGB-masked, deduplicated, split)
   - `jaguars_camera_trap_0226_duplicates` (segmented RGB-masked, with duplicates, split)
   - `jaguars_camera_trap_0226_raw` (raw full image, deduplicated, split)
   - `jaguars_camera_trap_0226_masked` (binary mask image, deduplicated, split)

In [ ]:
print("=" * 70)
print("STEP 7: Hugging Face Publishing")
print("=" * 70)

# ---------- auth ----------
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    login()  # interactive fallback

dataset = fo.load_dataset(DATASET_NAME)
image_view = dataset.select_group_slices("image") if dataset.group_field else dataset

# ---------- 7a: full dataset to HF (for manual review) ----------
run_export(
    dataset_name=DATASET_NAME,
    variants=["master"],
    export_targets=["huggingface"],
    huggingface_repo=HF_REPO_FULL_FIFTYONE,
    export_base_dir=EXPORT_BASE_DIR,
    segmentation_field=SEGMENTATION_FIELD,
    dedup_field=DEDUP_FIELD,
    verbose=VERBOSE,
    dry_run=False,
    summary_location=EXPORT_BASE_DIR / "hf_full_export_summary.json",
)
print(f"✓ Published full FiftyOne dataset: {HF_REPO_FULL_FIFTYONE}")

if not PUBLISH_DERIVED_AFTER_REVIEW:
    print("\nManual review gate is active.")
    print("Set PUBLISH_DERIVED_AFTER_REVIEW = True and rerun this cell after manual review.")
else:
    # ---------- helper utils for derived image datasets ----------
    def _detections_from_sample(sample, seg_field):
        dets_obj = sample.get(seg_field, None)
        if dets_obj is None or getattr(dets_obj, "detections", None) is None:
            return []
        return [d for d in dets_obj.detections if getattr(d, "bounding_box", None) is not None]

    def _resize_binary_mask(mask_array, target_w, target_h):
        m = Image.fromarray((mask_array > 0).astype(np.uint8) * 255)
        m = m.resize((target_w, target_h), resample=Image.NEAREST)
        return (np.array(m) > 0)

    def _union_mask(sample, seg_field, img_w, img_h):
        full = np.zeros((img_h, img_w), dtype=bool)
        for det in _detections_from_sample(sample, seg_field):
            bbox = det.bounding_box
            if bbox is None or len(bbox) != 4:
                continue

            x, y, w, h = bbox
            x0 = max(0, int(round(x * img_w)))
            y0 = max(0, int(round(y * img_h)))
            x1 = min(img_w, int(round((x + w) * img_w)))
            y1 = min(img_h, int(round((y + h) * img_h)))
            bw = max(0, x1 - x0)
            bh = max(0, y1 - y0)
            if bw == 0 or bh == 0:
                continue

            det_mask = getattr(det, "mask", None)
            if det_mask is None:
                full[y0:y1, x0:x1] = True
                continue

            arr = np.array(det_mask)
            if arr.ndim != 2:
                full[y0:y1, x0:x1] = True
                continue

            local = _resize_binary_mask(arr, bw, bh)
            full[y0:y1, x0:x1] |= local

        return full

    def _render_variant_image(sample, seg_field, mode):
        img = Image.open(sample.filepath).convert("RGB")
        arr = np.array(img)
        h, w = arr.shape[:2]
        mask = _union_mask(sample, seg_field, w, h)

        if mode == "raw":
            return img, ".jpg"
        if mode == "rgb_masked":
            out = np.zeros_like(arr)
            out[mask] = arr[mask]
            return Image.fromarray(out), ".png"
        if mode == "binary_mask":
            out = np.zeros((h, w), dtype=np.uint8)
            out[mask] = 255
            return Image.fromarray(out, mode="L"), ".png"

        raise ValueError(f"Unknown render mode: {mode}")

    def _clone_and_render(view, dataset_name, media_root, mode, seg_field):
        if fo.dataset_exists(dataset_name):
            fo.delete_dataset(dataset_name)
        ds = view.clone(dataset_name)

        out_dir = media_root / dataset_name
        out_dir.mkdir(parents=True, exist_ok=True)

        for sample in ds.iter_samples(progress=True):
            rendered, ext = _render_variant_image(sample, seg_field, mode)
            target = out_dir / f"{sample.id}{ext}"
            rendered.save(target)
            sample.filepath = str(target)
            sample.save()

        return ds

    def _push_private(dataset_or_view, repo_name):
        push_to_hub(
            dataset_or_view,
            repo_name=repo_name,
            dataset_type=fo.types.FiftyOneDataset,
            private=True,
        )
        print(f"✓ Published private HF dataset: {repo_name}")

    # ---------- 7b: define curated views ----------
    segmented_view = image_view.exists(SEGMENTATION_FIELD)
    segmented_dedup_view = segmented_view.match(F(DEDUP_FIELD) != True) if DEDUP_FIELD in image_view.get_field_schema() else segmented_view
    raw_dedup_view = image_view.match(F(DEDUP_FIELD) != True) if DEDUP_FIELD in image_view.get_field_schema() else image_view

    print("View sizes:")
    print(f"  segmented+dedup: {len(segmented_dedup_view)}")
    print(f"  segmented+duplicates: {len(segmented_view)}")
    print(f"  raw+dedup: {len(raw_dedup_view)}")

    # ---------- 7c: materialize transformed datasets ----------
    derived_media_root = EXPORT_BASE_DIR / "derived_media"
    derived_media_root.mkdir(parents=True, exist_ok=True)

    ds_segmented_dedup = _clone_and_render(
        segmented_dedup_view,
        dataset_name=f"{DATASET_NAME}_segmented_dedup_rgb",
        media_root=derived_media_root,
        mode="rgb_masked",
        seg_field=SEGMENTATION_FIELD,
    )
    ds_segmented_with_dup = _clone_and_render(
        segmented_view,
        dataset_name=f"{DATASET_NAME}_segmented_with_dup_rgb",
        media_root=derived_media_root,
        mode="rgb_masked",
        seg_field=SEGMENTATION_FIELD,
    )
    ds_raw_dedup = _clone_and_render(
        raw_dedup_view,
        dataset_name=f"{DATASET_NAME}_raw_dedup",
        media_root=derived_media_root,
        mode="raw",
        seg_field=SEGMENTATION_FIELD,
    )
    ds_binary_mask_dedup = _clone_and_render(
        segmented_dedup_view,
        dataset_name=f"{DATASET_NAME}_segmented_dedup_binary",
        media_root=derived_media_root,
        mode="binary_mask",
        seg_field=SEGMENTATION_FIELD,
    )

    # ---------- 7d: push derived datasets ----------
    _push_private(ds_segmented_dedup, HF_REPO_SEGMENTED_DEDUP)
    _push_private(ds_segmented_with_dup, HF_REPO_SEGMENTED_WITH_DUP)
    _push_private(ds_raw_dedup, HF_REPO_RAW_DEDUP)
    _push_private(ds_binary_mask_dedup, HF_REPO_MASK_BINARY_DEDUP)

    print("\n✓ All requested private datasets published to Hugging Face")

STEP 7: Export Dataset Variants
01:53:45 - jid_logger.ingestion.export.export - INFO - Starting export for dataset: JID_Master_Dataset
01:53:45 - jid_logger.ingestion.export.export - INFO - Exporting variant 'master' (1171 samples)
01:53:45 - jid_logger.ingestion.export.export - INFO -   -> Exporting to disk at /sc/home/philipp.kolbe/JID/camera-trap-footage/camera-trap-footage/data/intermediate/v1/fo_jaguars/exports/master
Exporting samples...
 100% |██████████████████| 1401/1401 [52.9s elapsed, 0s remaining, 63.5 docs/s]      
Exporting frames...
 100% |████████████████████████| 0/0 [258.9us elapsed, ? remaining, ? docs/s] 
01:54:39 - jid_logger.ingestion.export.export - INFO - Exporting variant 'segmented_deduplicated' (1065 samples)
01:54:39 - jid_logger.ingestion.export.export - INFO -   -> Exporting to disk at /sc/home/philipp.kolbe/JID/camera-trap-footage/camera-trap-footage/data/intermediate/v1/fo_jaguars/exports/segmented_deduplicated
Exporting samples...
 100% |███████████████

## Pipeline Summary

Display overall statistics and results from the ingestion pipeline.

In [ ]:
check_dataset_state(DATASET_NAME)

dataset = fo.load_dataset(DATASET_NAME)
img_view = dataset.select_group_slices("image") if dataset.group_field else dataset

print("\nQuick tag/field summary:")
print(f"  filter_count tags: {len(img_view.match_tags(SEGMENTATION_TAG_COUNT))}")
print(f"  filter_quality tags: {len(img_view.match_tags(SEGMENTATION_TAG_QUALITY))}")
if DEDUP_FIELD in img_view.get_field_schema():
    print(f"  is_duplicate=True: {len(img_view.match(F(DEDUP_FIELD) == True))}")
else:
    print("  is_duplicate field not found")

if SPLIT_FIELD in img_view.get_field_schema():
    vals = img_view.count_values(SPLIT_FIELD)
    print(f"  split distribution ({SPLIT_FIELD}): {vals}")
else:
    print(f"  split field not found: {SPLIT_FIELD}")

## Next Steps (Manual Review on Non-Cluster Machine)

Load `jaguars_camera_trap_0226_fiftyone` in FiftyOne UI and review these views before final exports:
- Samples tagged `filter_count` (wrong detection count)
- Samples tagged `filter_quality` (segmentation quality issues)
- Samples with `is_duplicate == True`
- Split distribution by `closed_set_split` (or `split`)

After corrections are synced, rerun Step 7 to republish the four derived private datasets.